##### Reduced-Rank Regression (RRR) Model

This notebook provides examples of how to configure, prepare data, train, and evaluate RRR models.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint

from models.decoders import ReducedRankDecoder

from utils.config_utils import config_from_kwargs, update_config
from utils.data_loader_utils import SingleSessionDataModule
from utils.eval_utils import eval_model
from utils.utils import set_seed

%cd neural_decoding

/burg/home/yz4123/.conda/envs/decoding/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# Load config
kwargs = {"model": "include:src/configs/decoder.yaml"}
config = config_from_kwargs(kwargs)
config = update_config("src/configs/decoder.yaml", config)

if target in REGRESSION:
    config = update_config("src/configs/reg_trainer.yaml", config)
elif target in CLASSIFICATION:
    config = update_config("src/configs/clf_trainer.yaml", config)
else:
    raise NotImplementedError

set_seed(config.seed)

seed set to 42


In [4]:
# Default data configurations
BINSIZE = 0.02
LENGTH = 2.
CLASSIFICATION = ["choice"]
REGRESSION = ["wheel-speed", "whisker-motion-energy", "prior"]

OUTPUT_SIZE_LOOKUP = {
    "choice": 2, 
    "prior": 1, 
    "wheel-speed": int(LENGTH / BINSIZE), 
    "whisker-motion-energy": int(LENGTH / BINSIZE),
}

In [5]:
# Configure parameters
base_path = "/burg/stats/users/yz4123/Downloads/"
eid = "754b74d5-7a06-4004-ae0c-72a10b6ed2e6"
target = "choice"  # options: choice, wheel-speed, etc.
region = "all"
method = "reduced_rank"
n_workers = 1
search = False  # Whether to perform hyperparameter sweep
use_nlb = False
bin_size = 20
fold_idx = 0

In [10]:
# Update config
config["dirs"]["data_dir"] = Path(base_path) / config.dirs.data_dir
save_path = Path(base_path) / config.dirs.output_dir / eid / target / method / region
ckpt_path = Path(base_path) / config.dirs.checkpoint_dir / eid / target / method / region
os.makedirs(save_path, exist_ok=True)
os.makedirs(ckpt_path, exist_ok=True)

config["eid"] = eid
config["target"] = target
config["region"] = region
config["model"]["output_size"] = OUTPUT_SIZE_LOOKUP[target]
config["training"]["device"] = torch.device(
    "cuda" if torch.cuda.is_available() and config.training.device == "gpu" else "cpu"
)
config["data"]["use_nlb"] = use_nlb
config["data"]["bin_size"] = bin_size
config["data"]["fold_idx"] = fold_idx

In [18]:
# Set up data module
dm = SingleSessionDataModule(config)
dm.update_config()
dm.setup()

config["training"]["total_steps"] = config["training"]["num_epochs"] * len(dm.train)

In [19]:
# Set up model
model = ReducedRankDecoder(config)
checkpoint_callback = ModelCheckpoint(
    monitor=config.training.metric,
    mode=config.training.mode,
    dirpath=ckpt_path
)
trainer = Trainer(
    max_epochs=100,
    callbacks=[checkpoint_callback],
    enable_progress_bar=config.training.enable_progress_bar,
    check_val_every_n_epoch=10,
    devices=1,
    strategy="auto",
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [20]:
# Train model
trainer.fit(model, datamodule=dm)

/burg/home/yz4123/.conda/envs/decoding/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /burg/stats/users/yz4123/Downloads/checkpoints/754b74d5-7a06-4004-ae0c-72a10b6ed2e6/choice/reduced_rank/all exists and is not empty.

  | Name         | Type               | Params | Mode 
------------------------------------------------------------
0 | metric       | MulticlassAccuracy | 0      | train
1 | loss         | CrossEntropyLoss   | 0      | train
  | other params | n/a                | 115 K  | n/a  
------------------------------------------------------------
115 K     Trainable params
0         Non-trainable params
115 K     Total params
0.460     Total estimated model params size (MB)
2         Modules in train mode
0         Modules in eval mode
SLURM auto-requeueing enabled. Setting signal handlers.


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/burg/home/yz4123/.conda/envs/decoding/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

/burg/home/yz4123/.conda/envs/decoding/lib/python3.11/site-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 16. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


...

Epoch 9: 100%|██████████| 37/37 [00:10<00:00,  3.41it/s, v_num=2786407, loss=20.00, val_metric=0.643, val_loss=438.0]

/burg/home/yz4123/.conda/envs/decoding/lib/python3.11/site-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 4. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Epoch 99: 100%|██████████| 37/37 [00:10<00:00,  3.41it/s, v_num=2786407, loss=0.0195, val_metric=0.726, val_loss=0.514]  

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 37/37 [00:10<00:00,  3.41it/s, v_num=2786407, loss=0.0195, val_metric=0.726, val_loss=0.514]


In [21]:
# Load model for evaluation
model = model.__class__.load_from_checkpoint(
    checkpoint_callback.best_model_path, config=config
)
train_dataset, test_dataset = dm.train, dm.test

In [22]:
# Evaluate model
metric, test_pred, test_y, test_prob = eval_model(
    train_dataset,
    test_dataset,
    model.cpu(),
    target=config["model"]["target"],
    model_class=method,
    use_nlb=use_nlb,
    bin_size=bin_size,
    beh_name=target,
)
print(f"Decoding results for {eid}: {metric}")

Decoding results for 754b74d5-7a06-4004-ae0c-72a10b6ed2e6: 0.7738095238095238


**Note:**
- The multi-session and multi-region RRR models follow the same procedure as the single-session case. 
- The main differences are in: 
    - Set up data config to load multiple sessions or regions
    - Use the appropriate model class and evaluation function 
- For details, see: 
    - `2_decode_multi_session.py`
    - `3_decode_multi_region.py`